# Module B - Technical Agent: Notebook 01 (Baselines)
## FXAlphaLab: Alpha Hurdle and Baseline ML Stack

This notebook establishes the baseline benchmark that all advanced models in notebook 02 must beat.

Scope is intentionally strict:
- Naive random walk (Alpha Hurdle)
- Random Forest + Bayesian hyperparameter search
- XGBoost + Bayesian hyperparameter search

No additional models are included by design.

In [5]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from skopt import BayesSearchCV
import xgboost as xgb
import shap
import joblib

from src.agents.technical.features import (
    load_pair, add_features, get_feature_names, PAIRS, TIMEFRAMES
)

MODEL_DIR = Path("../models/trained")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load all data
data = {}
for pair in PAIRS:
    data[pair] = {}
    for tf in TIMEFRAMES:
        df = load_pair(pair, tf)
        df = add_features(df)
        data[pair][tf] = df

print("✓ Data loaded successfully")
print(f"  Pairs     : {PAIRS}")
print(f"  Timeframes: {TIMEFRAMES}")
print(f"  Datasets  : {sum(len(v) for v in data.values())}")

✓ Data loaded successfully
  Pairs     : ['EURUSDm', 'GBPUSDm', 'USDJPYm', 'USDCHFm']
  Timeframes: ['D1', 'H4', 'H1']
  Datasets  : 12


## Validation Protocol (Walk-Forward with Purge and Embargo)

We use expanding-window walk-forward validation with explicit purge and embargo to preserve information-set discipline.

Rationale (required):
- Deep, G., Deep, A. & Lamptey, W. (2025). *Interpretable Hypothesis-Driven Trading: A Rigorous Walk-Forward Validation Framework for Market Microstructure Signals.* Texas Tech University.
- The protocol avoids lookahead bias and optimistic leakage common in random splits.

Implementation choices:
- Expanding training window
- Fixed-size test window
- Purge observations near split boundary from train
- Embargo gap after train boundary before test starts

Primary decision metric is directional accuracy (Hit Ratio). RMSE, MAPE, and Sharpe are reported for complete benchmarking continuity.

In [9]:
from dataclasses import dataclass
from typing import Callable

from sklearn.metrics import mean_squared_error
from skopt.space import Integer, Real

FEATURES = get_feature_names()
RANDOM_STATE = 42

# Set FAST_MODE=True for rapid iteration; set to False for final full benchmark run.
FAST_MODE = True

if FAST_MODE:
    OUTER_MIN_TRAIN = 600
    OUTER_TEST_SIZE = 300
    OUTER_PURGE = 5
    OUTER_EMBARGO = 5
    INNER_SPLITS = 2
    RF_N_ITER_FOLD = 4
    RF_N_ITER_FINAL = 10
    XGB_N_ITER_FOLD = 5
    XGB_N_ITER_FINAL = 10
    RUN_SHAP = False
    SHAP_MAX_SAMPLES = 300
else:
    OUTER_MIN_TRAIN = 600
    OUTER_TEST_SIZE = 120
    OUTER_PURGE = 5
    OUTER_EMBARGO = 5
    INNER_SPLITS = 3
    RF_N_ITER_FOLD = 10
    RF_N_ITER_FINAL = 10
    XGB_N_ITER_FOLD = 10
    XGB_N_ITER_FINAL = 10
    RUN_SHAP = True
    SHAP_MAX_SAMPLES = 1200

ANNUALIZATION = {
    "D1": 252,
    "H4": 252 * 6,
    "H1": 252 * 24,
}


@dataclass
class FoldResult:
    y_true: np.ndarray
    y_pred: np.ndarray
    ret_true: np.ndarray
    ret_pred: np.ndarray


def make_expanding_splits(
    n_samples: int,
    min_train_size: int,
    test_size: int,
    step_size: int | None = None,
    purge_size: int = 1,
    embargo_size: int = 1,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Create chronological expanding splits with explicit purge and embargo."""
    if step_size is None:
        step_size = test_size

    splits = []
    split_train_end = min_train_size

    while True:
        train_end_effective = split_train_end - purge_size
        test_start = split_train_end + embargo_size
        test_end = test_start + test_size

        if train_end_effective <= 0 or test_end > n_samples:
            break

        train_idx = np.arange(0, train_end_effective)
        test_idx = np.arange(test_start, test_end)
        splits.append((train_idx, test_idx))
        split_train_end += step_size

    return splits


def make_inner_walkforward_cv(
    n_train: int,
    n_splits: int = INNER_SPLITS,
    min_train_frac: float = 0.6,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Inner walk-forward CV used by BayesSearchCV (still time-ordered)."""
    if n_train < 80:
        pivot = max(20, int(n_train * 0.7))
        train_idx = np.arange(0, pivot)
        valid_idx = np.arange(pivot, n_train)
        return [(train_idx, valid_idx)]

    start = int(n_train * min_train_frac)
    remaining = n_train - start
    fold_size = max(20, remaining // n_splits)

    cv_splits = []
    train_end = start
    for _ in range(n_splits):
        valid_start = train_end
        valid_end = min(n_train, valid_start + fold_size)
        if valid_end - valid_start < 10:
            break
        tr_idx = np.arange(0, valid_start)
        va_idx = np.arange(valid_start, valid_end)
        cv_splits.append((tr_idx, va_idx))
        train_end = valid_end

    if not cv_splits:
        pivot = max(20, int(n_train * 0.7))
        cv_splits = [(np.arange(0, pivot), np.arange(pivot, n_train))]

    return cv_splits


def safe_mape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-8) -> float:
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def sharpe_ratio(returns: np.ndarray, periods_per_year: int) -> float:
    vol = np.std(returns, ddof=1)
    if vol == 0 or np.isnan(vol):
        return 0.0
    return float(np.mean(returns) / vol * np.sqrt(periods_per_year))


def compute_metrics(fold_results: list[FoldResult], periods_per_year: int) -> dict:
    y_true = np.concatenate([f.y_true for f in fold_results])
    y_pred = np.concatenate([f.y_pred for f in fold_results])
    ret_true = np.concatenate([f.ret_true for f in fold_results])
    ret_pred = np.concatenate([f.ret_pred for f in fold_results])

    strategy_returns = (2 * y_pred - 1) * ret_true

    return {
        "rmse": float(np.sqrt(mean_squared_error(ret_true, ret_pred))),
        "mape": safe_mape(ret_true, ret_pred),
        "hit_ratio": float(accuracy_score(y_true, y_pred)),
        "sharpe": sharpe_ratio(strategy_returns, periods_per_year),
        "n_obs": int(len(y_true)),
    }


def evaluate_model_walkforward(
    df: pd.DataFrame,
    tf: str,
    predictor: Callable[[np.ndarray, np.ndarray, np.ndarray], np.ndarray],
    min_train_size: int = OUTER_MIN_TRAIN,
    test_size: int = OUTER_TEST_SIZE,
    purge_size: int = OUTER_PURGE,
    embargo_size: int = OUTER_EMBARGO,
) -> tuple[list[FoldResult], list[tuple[np.ndarray, np.ndarray]]]:
    X = df[FEATURES].to_numpy()
    y = df["target"].to_numpy().astype(int)
    next_ret = df["log_ret"].shift(-1).fillna(0.0).to_numpy()

    n_samples = len(df)
    if n_samples < min_train_size + test_size + purge_size + embargo_size:
        min_train_size = max(200, int(n_samples * 0.5))
        test_size = max(40, int(n_samples * 0.15))

    splits = make_expanding_splits(
        n_samples=n_samples,
        min_train_size=min_train_size,
        test_size=test_size,
        purge_size=purge_size,
        embargo_size=embargo_size,
    )

    fold_results: list[FoldResult] = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        y_pred = predictor(X_train, y_train, X_test).astype(int)

        # Direction-only models are mapped to a return proxy using train fold average magnitude.
        avg_abs_ret = float(np.mean(np.abs(next_ret[train_idx])))
        ret_pred = (2 * y_pred - 1) * avg_abs_ret
        ret_true = next_ret[test_idx]

        fold_results.append(
            FoldResult(
                y_true=y_test,
                y_pred=y_pred,
                ret_true=ret_true,
                ret_pred=ret_pred,
            )
        )

    return fold_results, splits


def save_shap_summary(
    model, X_df: pd.DataFrame, out_path: Path, max_samples: int | None = None
) -> None:
    if max_samples is None:
        max_samples = SHAP_MAX_SAMPLES

    sample = X_df if len(X_df) <= max_samples else X_df.sample(max_samples, random_state=RANDOM_STATE)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(sample)
    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    else:
        shap_values_to_plot = shap_values

    plt.figure(figsize=(11, 6))
    shap.summary_plot(shap_values_to_plot, sample, show=False, max_display=15)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


print("✓ Walk-forward utilities ready")
print(f"✓ Feature count: {len(FEATURES)}")
print(f"✓ FAST_MODE: {FAST_MODE}")

✓ Walk-forward utilities ready
✓ Feature count: 15
✓ FAST_MODE: True


## Model 1: Naive Random Walk (Alpha Hurdle)

Architecture rationale (required citation):
- Meese, R. & Rogoff, K. (1983). *Empirical Exchange Rate Models of the Seventies: Do They Fit Out of Sample?* Journal of International Economics, 14(1-2), 3-24.

Interpretation:
- This baseline predicts next direction as current direction.
- It defines the minimum hurdle for any model claiming predictive alpha.
- If RF/XGBoost cannot beat this baseline, they are rejected.

In [7]:
def random_walk_predictor(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray) -> np.ndarray:
    # Using log_ret sign from the last available observation in each test row feature vector.
    # FEATURES[0] is log_ret by construction from features.py
    log_ret_col = FEATURES.index("log_ret")
    return (X_test[:, log_ret_col] > 0).astype(int)

random_walk_rows = []

for pair in PAIRS:
    for tf in TIMEFRAMES:
        df = data[pair][tf].copy()
        folds, splits = evaluate_model_walkforward(
            df=df,
            tf=tf,
            predictor=random_walk_predictor,
        )
        metrics = compute_metrics(folds, periods_per_year=ANNUALIZATION[tf])
        random_walk_rows.append({
            "model": "naive_random_walk",
            "pair": pair,
            "timeframe": tf,
            "folds": len(splits),
            **metrics,
        })

random_walk_df = pd.DataFrame(random_walk_rows)
print("✓ Random walk baseline completed")
display(random_walk_df.head())

✓ Random walk baseline completed


,model,pair,timeframe,folds,rmse,mape,hit_ratio,sharpe,n_obs
0,naive_random_walk,EURUSDm,D1,2,0.005258,549.831561,0.495000,-0.437040,600
1,naive_random_walk,EURUSDm,H4,23,0.002347,54810.153153,0.488551,-0.698911,6900
2,naive_random_walk,EURUSDm,H1,100,0.001192,48600.054736,0.485100,-1.183903,30000
3,naive_random_walk,GBPUSDm,D1,2,0.005459,62667.175896,0.481667,-0.663892,600
4,naive_random_walk,GBPUSDm,H4,23,0.002652,41802.022870,0.487681,-0.280295,6900


## Model 2: Random Forest + Bayesian Hyperparameter Search

Architecture rationale (required citation):
- Guyard, K.C. & Deriaz, M. (2024). *Predicting Foreign Exchange EURUSD Direction Using Machine Learning.* MLMI 2024, ACM.

Hyperparameter rationale:
- Bayesian search is used to efficiently explore non-linear parameter interactions with fewer evaluations than exhaustive grids.
- Search space emphasizes depth, tree count, minimum split/leaf constraints, and max feature proportion to control overfitting in noisy FX series.

Protocol constraints:
- Inner search uses chronological walk-forward folds only.
- Final evaluation uses the outer walk-forward with purge and embargo.

In [10]:
RF_SPACE = (
    {
        "n_estimators": Integer(50, 200),
        "max_depth": Integer(3, 8),
        "min_samples_split": Integer(2, 20),
        "min_samples_leaf": Integer(1, 10),
        "max_features": Real(0.35, 1.0),
    }
    if FAST_MODE
    else {
        "n_estimators": Integer(120, 700),
        "max_depth": Integer(3, 16),
        "min_samples_split": Integer(2, 30),
        "min_samples_leaf": Integer(1, 20),
        "max_features": Real(0.35, 1.0),
    }
)


def rf_predictor(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray) -> np.ndarray:
    # Quick safe profile: in FAST_MODE use fixed RF params and skip Bayesian search.
    if FAST_MODE:
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            class_weight="balanced_subsample",
        )
        rf.fit(X_train, y_train)
        return rf.predict(X_test)

    cv_splits = make_inner_walkforward_cv(len(X_train), n_splits=INNER_SPLITS, min_train_frac=0.6)
    base = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    opt = BayesSearchCV(
        estimator=base,
        search_spaces=RF_SPACE,
        n_iter=RF_N_ITER_FOLD,
        cv=cv_splits,
        scoring="accuracy",
        n_jobs=1,
        random_state=RANDOM_STATE,
        refit=True,
    )
    opt.fit(X_train, y_train)
    return opt.best_estimator_.predict(X_test)


def train_final_rf(X: np.ndarray, y: np.ndarray) -> RandomForestClassifier:
    if FAST_MODE:
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            class_weight="balanced_subsample",
        )
        rf.fit(X, y)
        return rf

    cv_splits = make_inner_walkforward_cv(len(X), n_splits=INNER_SPLITS, min_train_frac=0.6)
    base = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    opt = BayesSearchCV(
        estimator=base,
        search_spaces=RF_SPACE,
        n_iter=RF_N_ITER_FINAL,
        cv=cv_splits,
        scoring="accuracy",
        n_jobs=1,
        random_state=RANDOM_STATE,
        refit=True,
    )
    opt.fit(X, y)
    return opt.best_estimator_


rf_rows = []

for pair in PAIRS:
    for tf in TIMEFRAMES:
        df = data[pair][tf].copy()

        folds, splits = evaluate_model_walkforward(
            df=df,
            tf=tf,
            predictor=rf_predictor,
            min_train_size=OUTER_MIN_TRAIN,
            test_size=OUTER_TEST_SIZE,
            purge_size=OUTER_PURGE,
            embargo_size=OUTER_EMBARGO,
        )
        metrics = compute_metrics(folds, periods_per_year=ANNUALIZATION[tf])

        X_all = df[FEATURES].to_numpy()
        y_all = df["target"].to_numpy().astype(int)
        rf_final = train_final_rf(X_all, y_all)

        model_path = MODEL_DIR / f"rf_{pair}_{tf}.pkl"
        joblib.dump(rf_final, model_path)

        shap_path = None
        if RUN_SHAP:
            shap_path = MODEL_DIR / f"shap_rf_{pair}_{tf}.png"
            save_shap_summary(rf_final, df[FEATURES], shap_path, max_samples=SHAP_MAX_SAMPLES)

        rf_rows.append(
            {
                "model": "random_forest_bayes" if not FAST_MODE else "random_forest_fixed",
                "pair": pair,
                "timeframe": tf,
                "folds": len(splits),
                "model_path": str(model_path),
                "shap_path": str(shap_path) if shap_path else "",
                **metrics,
            }
        )

rf_df = pd.DataFrame(rf_rows)
print("✓ Random Forest baseline completed")
display(rf_df.head())

✓ Random Forest baseline completed


,model,pair,timeframe,folds,model_path,shap_path,rmse,mape,hit_ratio,sharpe,n_obs
0,random_forest_fixed,EURUSDm,D1,3,..\models\trained\rf_EURUSDm_D1.pkl,,0.005216,631.520962,0.532222,0.310877,900
1,random_forest_fixed,EURUSDm,H4,24,..\models\trained\rf_EURUSDm_H4.pkl,,0.002299,56785.380156,0.506250,0.248567,7200
2,random_forest_fixed,EURUSDm,H1,101,..\models\trained\rf_EURUSDm_H1.pkl,,0.001182,48489.485385,0.506403,-0.159068,30300
3,random_forest_fixed,GBPUSDm,D1,3,..\models\trained\rf_GBPUSDm_D1.pkl,,0.005624,41976.570008,0.504444,-0.170595,900
4,random_forest_fixed,GBPUSDm,H4,24,..\models\trained\rf_GBPUSDm_H4.pkl,,0.002620,40083.519346,0.505556,0.061780,7200


## Model 3: XGBoost + Bayesian Hyperparameter Search

Architecture rationale (required citations):
- Guyard, K.C. & Deriaz, M. (2024): Bayesian optimization for FX directional models.
- Ozturk, C. (2025): XGBoost selected as primary ensemble due to superior classification and trading outcomes versus alternatives.

Hyperparameter rationale:
- Search focuses on depth/learning-rate/estimators and row-column subsampling to stabilize performance in heteroskedastic FX data.

Protocol constraints:
- Same walk-forward protocol and artifact policy as RF for strict comparability.

In [11]:
XGB_SPACE = {
    "n_estimators": Integer(100, 900),
    "max_depth": Integer(2, 10),
    "learning_rate": Real(0.01, 0.30, prior="log-uniform"),
    "subsample": Real(0.5, 1.0),
    "colsample_bytree": Real(0.4, 1.0),
    "gamma": Real(0.0, 5.0),
    "min_child_weight": Integer(1, 12),
}


def xgb_predictor(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray) -> np.ndarray:
    cv_splits = make_inner_walkforward_cv(len(X_train), n_splits=INNER_SPLITS, min_train_frac=0.6)
    base = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
    )
    opt = BayesSearchCV(
        estimator=base,
        search_spaces=XGB_SPACE,
        n_iter=XGB_N_ITER_FOLD,
        cv=cv_splits,
        scoring="accuracy",
        n_jobs=1,
        random_state=RANDOM_STATE,
        refit=True,
    )
    opt.fit(X_train, y_train)
    return opt.best_estimator_.predict(X_test)


def train_final_xgb(X: np.ndarray, y: np.ndarray) -> xgb.XGBClassifier:
    cv_splits = make_inner_walkforward_cv(len(X), n_splits=INNER_SPLITS, min_train_frac=0.6)
    base = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
    )
    opt = BayesSearchCV(
        estimator=base,
        search_spaces=XGB_SPACE,
        n_iter=XGB_N_ITER_FINAL,
        cv=cv_splits,
        scoring="accuracy",
        n_jobs=1,
        random_state=RANDOM_STATE,
        refit=True,
    )
    opt.fit(X, y)
    return opt.best_estimator_


xgb_rows = []

for pair in PAIRS:
    for tf in TIMEFRAMES:
        df = data[pair][tf].copy()

        folds, splits = evaluate_model_walkforward(
            df=df,
            tf=tf,
            predictor=xgb_predictor,
            min_train_size=OUTER_MIN_TRAIN,
            test_size=OUTER_TEST_SIZE,
            purge_size=OUTER_PURGE,
            embargo_size=OUTER_EMBARGO,
        )
        metrics = compute_metrics(folds, periods_per_year=ANNUALIZATION[tf])

        X_all = df[FEATURES].to_numpy()
        y_all = df["target"].to_numpy().astype(int)
        xgb_final = train_final_xgb(X_all, y_all)

        model_path = MODEL_DIR / f"xgb_{pair}_{tf}.pkl"
        joblib.dump(xgb_final, model_path)

        shap_path = None
        if RUN_SHAP:
            shap_path = MODEL_DIR / f"shap_xgb_{pair}_{tf}.png"
            save_shap_summary(xgb_final, df[FEATURES], shap_path, max_samples=SHAP_MAX_SAMPLES)

        xgb_rows.append(
            {
                "model": "xgboost_bayes",
                "pair": pair,
                "timeframe": tf,
                "folds": len(splits),
                "model_path": str(model_path),
                "shap_path": str(shap_path) if shap_path else "",
                **metrics,
            }
        )

xgb_df = pd.DataFrame(xgb_rows)
print("✓ XGBoost baseline completed")
display(xgb_df.head())

metrics_df = pd.concat([random_walk_df, rf_df, xgb_df], ignore_index=True)
metrics_df = metrics_df.sort_values(["pair", "timeframe", "model"]).reset_index(drop=True)

overall_df = (
    metrics_df
    .groupby("model", as_index=False)
    .agg(
        {
            "rmse": "mean",
            "mape": "mean",
            "hit_ratio": "mean",
            "sharpe": "mean",
            "n_obs": "sum",
        }
    )
    .sort_values("hit_ratio", ascending=False)
    .reset_index(drop=True)
)

metrics_csv_path = MODEL_DIR / "baseline_metrics_table.csv"
metrics_df.to_csv(metrics_csv_path, index=False)

overall_csv_path = MODEL_DIR / "baseline_metrics_overall.csv"
overall_df.to_csv(overall_csv_path, index=False)

print("\n✓ Metrics table saved")
print(f"  Detailed: {metrics_csv_path}")
print(f"  Overall : {overall_csv_path}")

print("\n=== Per pair/timeframe metrics ===")
display(metrics_df)

print("=== Overall model summary ===")
display(overall_df)

✓ XGBoost baseline completed


,model,pair,timeframe,folds,model_path,shap_path,rmse,mape,hit_ratio,sharpe,n_obs
0,xgboost_bayes,EURUSDm,D1,3,..\models\trained\xgb_EURUSDm_D1.pkl,,0.005293,633.624659,0.516667,-0.168395,900
1,xgboost_bayes,EURUSDm,H4,24,..\models\trained\xgb_EURUSDm_H4.pkl,,0.002300,56786.583634,0.498889,0.194884,7200
2,xgboost_bayes,EURUSDm,H1,101,..\models\trained\xgb_EURUSDm_H1.pkl,,0.001180,48489.447656,0.507789,0.085199,30300
3,xgboost_bayes,GBPUSDm,D1,3,..\models\trained\xgb_GBPUSDm_D1.pkl,,0.005598,41976.953819,0.504444,-0.026382,900
4,xgboost_bayes,GBPUSDm,H4,24,..\models\trained\xgb_GBPUSDm_H4.pkl,,0.002619,40083.732627,0.504028,0.032502,7200



✓ Metrics table saved
  Detailed: ..\models\trained\baseline_metrics_table.csv
  Overall : ..\models\trained\baseline_metrics_overall.csv

=== Per pair/timeframe metrics ===


,model,pair,timeframe,folds,rmse,mape,hit_ratio,sharpe,n_obs,model_path,shap_path
0,naive_random_walk,EURUSDm,D1,2,0.005258,549.831561,0.495000,-0.437040,600,NaN,NaN
1,random_forest_fixed,EURUSDm,D1,3,0.005216,631.520962,0.532222,0.310877,900,..\models\trained\rf_EURUSDm_D1.pkl,
2,xgboost_bayes,EURUSDm,D1,3,0.005293,633.624659,0.516667,-0.168395,900,..\models\trained\xgb_EURUSDm_D1.pkl,
3,naive_random_walk,EURUSDm,H1,100,0.001192,48600.054736,0.485100,-1.183903,30000,NaN,NaN
4,random_forest_fixed,EURUSDm,H1,101,0.001182,48489.485385,0.506403,-0.159068,30300,..\models\trained\rf_EURUSDm_H1.pkl,
5,xgboost_bayes,EURUSDm,H1,101,0.001180,48489.447656,0.507789,0.085199,30300,..\models\trained\xgb_EURUSDm_H1.pkl,
6,naive_random_walk,EURUSDm,H4,23,0.002347,54810.153153,0.488551,-0.698911,6900,NaN,NaN
7,random_forest_fixed,EURUSDm,H4,24,0.002299,56785.380156,0.506250,0.248567,7200,..\models\trained\rf_EURUSDm_H4.pkl,
8,xgboost_bayes,EURUSDm,H4,24,0.002300,56786.583634,0.498889,0.194884,7200,..\models\trained\xgb_EURUSDm_H4.pkl,
9,naive_random_walk,GBPUSDm,D1,2,0.005459,62667.175896,0.481667,-0.663892,600,NaN,NaN


=== Overall model summary ===


,model,rmse,mape,hit_ratio,sharpe,n_obs
0,xgboost_bayes,0.003221,44499.267711,0.507646,0.189542,153600
1,random_forest_fixed,0.003226,44499.851325,0.504680,0.228748,153600
2,naive_random_walk,0.003257,47556.308276,0.489095,-0.582053,150000


## Baseline Conclusion (For Notebook 02 Handoff)

This notebook establishes the Alpha Hurdle and baseline ensemble benchmarks across all 12 datasets.

Handoff artifacts produced:
- Trained Random Forest and XGBoost models in models/trained
- SHAP summary plots for RF and XGBoost in models/trained
- Detailed and overall metrics tables saved as CSV

Notebook 02 (deep learning) must exceed these baseline results, especially directional accuracy and risk-adjusted performance (Sharpe), to justify additional model complexity.